In [1]:
import platform

print(f"Machine Name: {platform.node()}")
# If it says something like 'colab-backend', you are successfully assigned!

Machine Name: DESKTOP-N7CAFKF


<!-- @format -->


In [9]:
# This was the cell causing the hang.
# VS Code often blocks the "Mount Google Drive" popup, causing it to freeze infinitely.
# We are skipping this and switching back to your LOCAL environment to guarantee it works.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<!-- @format -->

### Pre Processing Refinary


In [1]:
import pandas as pd
import re
import os


def normalize_amharic(text):
    if not isinstance(text, str):
        return ""
    # Standardizing homophones
    replacements = {
        "ሐ": "ሀ",
        "ኀ": "ሀ",
        "ኃ": "ሀ",
        "ሠ": "ሰ",
        "ዐ": "አ",
        "ዓ": "አ",
        "ፀ": "ጸ",
        "ፅ": "ጽ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    # Remove English noise
    text = re.sub(r"[a-zA-Z]", "", text)
    return re.sub(r"\s+", " ", text).strip()


# WE ARE USING YOUR LOCAL MACHINE INSTEAD OF COLAB
# Make sure your file is located exactly here:
file_path_local = r"D:\Amaharic-Sport-News-Curation\Amharic News Dataset.csv"

# Check if the file exists on your local drive
if os.path.exists(file_path_local):
    df = pd.read_csv(file_path_local)
    # Filter for Sports and take 1,100 to be safe
    sports_df = df[df["category"] == "ስፖርት"].head(1200).copy()
    sports_df["cleaned"] = sports_df["content"].apply(normalize_amharic)
    print(f"✅ Prepared {len(sports_df)} sports articles for curation.")
else:
    print(f"❌ ERROR: File not found at {file_path_local}!")

KeyError: 'content'

<!-- @format -->

---

### **Step 3: The Persistent Curation Loop**

This cell uses the Gemini API to format the content into Alpaca JSONL format, saving directly to your currently mounted Google Drive.


In [ ]:
import google.generativeai as genai
import json
import time
import os

# 1. Put your actual Gemini API Key here as a string between the quotes:
genai.configure(api_key="PASTE_YOUR_API_KEY_HERE")
model = genai.GenerativeModel("gemini-1.5-flash")

# 2. Saving the curated JSONL directly into your local workspace folder
output_path = r"D:\Amaharic-Sport-News-Curation\amharic_sports_curated.jsonl"


def curate_batch():
    # Load existing progress to avoid duplicates
    try:
        with open(output_path, "r", encoding="utf-8") as f:
            done_count = sum(1 for _ in f)
    except FileNotFoundError:
        done_count = 0

    print(f"Resuming from row {done_count}...")

    with open(output_path, "a", encoding="utf-8") as f:
        for i, row in sports_df.iloc[done_count:].iterrows():
            prompt = f"Read this Amharic news: {row['cleaned'][:1200]}. Generate 1 instruction and 1 response in Amharic. Format as JSON."
            try:
                response = model.generate_content(prompt)
                # Parse to ensure it conforms.
                json_data = json.loads(
                    response.text.replace("```json", "").replace("```", "").strip()
                )

                # Alpaca Format: Instruction, Input, Output
                entry = {
                    "instruction": json_data["instruction"],
                    "input": row["cleaned"],
                    "output": json_data["output"],
                }
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

                if (done_count + i) % 10 == 0:
                    print(f"Successfully curated {done_count + i} rows.")
            except Exception as e:
                print(f"Error on row {i}: {e}")
                continue
            time.sleep(1.2)  # Safety delay for API limits


curate_batch()

<!-- @format -->

---

### **Step 4: The Final Submission Check**

Run this final cell to verify your work.


In [ ]:
# Verify row count
with open(output_path, "r", encoding="utf-8") as f:
    total_rows = sum(1 for _ in f)

print(f"Total rows curated: {total_rows}")

# Preview the first entry
with open(output_path, "r", encoding="utf-8") as f:
    print("Sample Entry:")
    print(json.loads(f.readline()))